# Investigating the V02 checkpoint 893 Perfect Accuracy
v02/downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2/test_metrics.json

# 1. Hub drift 


The trainer_state.json from checkpoint-893 shows:

"epoch": 1.0 — F1 hit 1.0 after just 1 epoch of training
"eval_loss": 0.000118 — essentially zero on validation
"max_steps": 17860 — training was set for 20 epochs (893 steps × 20)
Achieving F1=1.0 on val AND test in 1 epoch with non-trivial training loss (0.23) is the smoking gun — the validation set is almost certainly leaking from (or identical to) the training data.

In [4]:
from huggingface_hub import hf_hub_download
import json

In [5]:
HUGGINGFACE_REPO_ID="Rogarcia18/symptoms_ner_v02_biobert"
DATASET_REPO_ID=HUGGINGFACE_REPO_ID
# Download and load id2label.json from the hub
id2label_path = hf_hub_download(
        repo_id=DATASET_REPO_ID,
        filename="id2label.json",
        repo_type="dataset"
    )
# Download and load label2id.json from the hub
label2id_path = hf_hub_download(
        repo_id=DATASET_REPO_ID,
        filename="label2id.json",
        repo_type="dataset"
    )
with open(id2label_path, "r") as f:
    id2label = json.load(f)
with open(label2id_path, "r") as f:
    label2id = json.load(f)

# Number of labels 
num_labels = len(id2label)

**Training, Test, and Validation Set Test Overlap**

In [7]:
# Step 1: Check for train/val/test text overlap on the Hub
from datasets import load_dataset

ds = load_dataset(DATASET_REPO_ID)
print("Sizes:", {k: len(v) for k, v in ds.items()})

train_texts = set(ds["train"]["text"])
val_texts   = set(ds["validation"]["text"])
test_texts  = set(ds["test"]["text"])

print(f"\nTrain ∩ Val:  {len(train_texts & val_texts)} overlapping texts")
print(f"Train ∩ Test: {len(train_texts & test_texts)} overlapping texts")
print(f"Val ∩ Test:   {len(val_texts & test_texts)} overlapping texts")

Sizes: {'train': 14288, 'validation': 1786, 'test': 1786}

Train ∩ Val:  0 overlapping texts
Train ∩ Test: 0 overlapping texts
Val ∩ Test:   0 overlapping texts


**input_ids overlap (duplicates at token level)**

In [8]:
# Step 2: Check input_ids overlap (catches same tokenized sequence even if text differs slightly)
train_ids = set(tuple(x) for x in ds["train"]["input_ids"])
val_ids   = set(tuple(x) for x in ds["validation"]["input_ids"])
test_ids  = set(tuple(x) for x in ds["test"]["input_ids"])

print(f"Train ∩ Val (input_ids):  {len(train_ids & val_ids)}")
print(f"Train ∩ Test (input_ids): {len(train_ids & test_ids)}")

Train ∩ Val (input_ids):  0
Train ∩ Test (input_ids): 0


In [16]:
ds["test"]

Dataset({
    features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
    num_rows: 1786
})

**3) Run the downloaded model on a few test examples and inspect**

In [17]:
# Step 3: Sanity-check the model is actually predicting, not cheating
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch
import os

# Local directory (resolved absolute path) for model loading
MODEL_PATH = os.path.abspath("downloaded_models/dmis-lab/biobert-base-cased-v1.1/run_2")
model = AutoModelForTokenClassification.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

model.eval()

# Pick a few test examples from the Hub
for i in range(3):
    example = ds["test"][i]
    inputs = {k: torch.tensor([example[k]]) for k in ["input_ids", "attention_mask", "token_type_ids"] if k in example}
    with torch.no_grad():
        logits = model(**inputs).logits
    preds = logits.argmax(-1)[0].tolist()
    gold  = example["token_label_ids"]
    tokens = tokenizer.convert_ids_to_tokens(example["input_ids"])
    
    print(f"\n--- Example {i} ---")
    for tok, g, p in zip(tokens, gold, preds):
        if g != -100:
            match = "✓" if g == p else "✗"
            print(f"  {match} {tok:25s}  gold={id2label[str(g)]:20s}  pred={id2label[str(p)]}")


--- Example 0 ---
  ✓ the                        gold=O                     pred=O
  ✓ patient                    gold=O                     pred=O
  ✓ does                       gold=O                     pred=O
  ✓ not                        gold=O                     pred=O
  ✓ have                       gold=O                     pred=O
  ✓ s                          gold=B-SYMPTOM_NEG         pred=B-SYMPTOM_NEG
  ✓ ##nee                      gold=I-SYMPTOM_NEG         pred=I-SYMPTOM_NEG
  ✓ ##zing                     gold=I-SYMPTOM_NEG         pred=I-SYMPTOM_NEG
  ✓ .                          gold=O                     pred=O

--- Example 1 ---
  ✓ the                        gold=O                     pred=O
  ✓ patient                    gold=O                     pred=O
  ✓ is                         gold=O                     pred=O
  ✓ free                       gold=O                     pred=O
  ✓ of                         gold=O                     pred=O
  ✓ le          

### Conclusion up to this point:

- No overlapping texts and no input ids between the splits. We can rule out the most common form of data leakage.

## V01 Dataset vs V02 Dataset

In [18]:
ds_v01 = load_dataset("Rogarcia18/symptoms_ner_v01_biobert")
print("v01 sizes:", {k: len(v) for k, v in ds_v01.items()})
print("v02 sizes:", {k: len(v) for k, v in ds.items()})

# Same texts?
print(f"\nv01 train == v02 train (texts): {set(ds_v01['train']['text']) == set(ds['train']['text'])}")
print(f"v01 test  == v02 test  (texts): {set(ds_v01['test']['text']) == set(ds['test']['text'])}")

# Same columns?
print(f"\nv01 columns: {ds_v01['train'].column_names}")
print(f"v02 columns: {ds['train'].column_names}")

Generating test split: 100%|██████████| 1786/1786 [00:00<00:00, 685928.66 examples/s]


v01 sizes: {'train': 14288, 'validation': 1786, 'test': 1786}
v02 sizes: {'train': 14288, 'validation': 1786, 'test': 1786}

v01 train == v02 train (texts): True
v01 test  == v02 test  (texts): True

v01 columns: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids']
v02 columns: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids']


**Quantify how templated the data is**

In [19]:
import re

# Extract sentence structure (replace symptom with placeholder)
def template_of(text):
    # crude: lowercase, collapse symptom spans
    t = text.lower().strip().rstrip(".")
    words = t.split()
    # Find where "the patient" preamble ends — everything after is likely symptom
    for i, w in enumerate(words):
        if w not in ("the", "patient", "is", "has", "does", "not", "have", 
                     "reports", "denies", "describes", "having", "no", 
                     "free", "of", "presenting", "with", "exhibits",
                     "without", "a", "an"):
            return " ".join(words[:i]) + " [SYMPTOM]"
    return " ".join(words)

templates = [template_of(t) for t in ds["train"]["text"]]
from collections import Counter
template_counts = Counter(templates)
print(f"Unique templates: {len(template_counts)} out of {len(templates)} examples")
for t, c in template_counts.most_common(10):
    print(f"  {c:5d}x  '{t}'")

Unique templates: 15 out of 14288 examples
   2157x  ' [SYMPTOM]'
   2118x  'patient [SYMPTOM]'
   1428x  'no [SYMPTOM]'
    731x  'denies [SYMPTOM]'
    728x  'patient reports [SYMPTOM]'
    728x  'patient has no [SYMPTOM]'
    724x  'patient is [SYMPTOM]'
    721x  'patient denies [SYMPTOM]'
    720x  'the patient is free of [SYMPTOM]'
    719x  'not [SYMPTOM]'
